In [ ]:
import pickle
import numpy as np
import pandas as pd
import tifffile as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import skimage

In [ ]:
keys = ['HKT ANCA', 'HKT CTRL 1']

In [ ]:
glomerulus_paths = [] # paths to glomerulosclerosis segmentation results for each sample
pixel_clustering_paths = [] # paths to pixel clustering results for each sample

In [ ]:
fibrosis_paths = [] # paths to color deconvolution results for each sample

In [ ]:
non_empty_pixels_dict = {}

i = 0
for key in tqdm(keys):
    non_empty_pixels_dict[key] = pd.read_csv(pixel_clustering_paths[i] + '/non_empty_pixels.csv').drop(columns=['Unnamed: 0'])
    i+=1 

In [ ]:
wsi_cluster_labeled_dict = {}
wsi_glomerulus_labeled_dict = {}
wsi_trichrome_blue_dict = {}
cluster_color_map_dict = {}
i = 0
for key in keys:
    with open(pixel_clustering_paths[i] + '/wsi_labeled.pickle', 'rb') as f: 
        wsi_cluster_labeled_dict[key] = pickle.load(f) 
    with open(pixel_clustering_paths[i] + '/cluster_color_dict.pickle', 'rb') as f: 
            cluster_color_map_dict[key] = pickle.load(f) 
    wsi_glomerulus_labeled_dict[key] = tf.imread(glomerulus_paths[i] + '/mask_registered.tif')
    wsi_trichrome_blue_dict[key] = 255-tf.imread(fibrosis_paths[i] + '/trichrome_blue_non_wsi.tif')
    i+=1 

In [ ]:
wsi_fibrosis_dict = {}
for key, image in wsi_trichrome_blue_dict.items():
    wsi_fibrosis_dict[key] = (image > skimage.filters.threshold_multiotsu(image, classes=2)[-1]).astype(int)

In [ ]:
pixel_cluster_label_df_dict = {}

In [ ]:

for key in tqdm(keys): 
    image_stack = np.stack([wsi_cluster_labeled_dict[key], wsi_glomerulus_labeled_dict[key]])

    num_images, height, width = image_stack.shape
    num_pixels = height * width
    image_stack_flattened = image_stack.reshape(num_images, num_pixels).T
    pixels_df = pd.DataFrame(image_stack_flattened, columns=['pixel_som_cluster', 'glomerulus_label'])
    x = image_stack.shape[1]
    y = image_stack.shape[2]
        
    pixel_range = [i for i in range(x)]
    pixels_df['X'] = np.repeat(pixel_range, y)
    pixels_df['Y'] = [i for i in range(y)] * x
    coords = pd.MultiIndex.from_frame(
    non_empty_pixels_dict[key][['X', 'Y']])

    pixels_df = pixels_df[
        pd.MultiIndex.from_frame(pixels_df[['X', 'Y']]).isin(coords)
    ].copy()

    pixel_cluster_label_df_dict[key] = pixels_df


In [ ]:
pixel_cluster_fibrosis_df_dict = {}

In [ ]:

for key in tqdm(keys): 
    image_stack = np.stack([wsi_cluster_labeled_dict[key], wsi_fibrosis_dict[key]])

    num_images, height, width = image_stack.shape
    num_pixels = height * width
    image_stack_flattened = image_stack.reshape(num_images, num_pixels).T
    pixels_df = pd.DataFrame(image_stack_flattened, columns=['pixel_som_cluster', 'fibrosis'])
    x = image_stack.shape[1]
    y = image_stack.shape[2]
        
    pixel_range = [i for i in range(x)]
    pixels_df['X'] = np.repeat(pixel_range, y)
    pixels_df['Y'] = [i for i in range(y)] * x
    non_empty_pixels_dict[key][['X', 'Y']]
    
    pixels_df = pixels_df[
        pd.MultiIndex.from_frame(pixels_df[['X', 'Y']]).isin(coords)
    ].copy()
    pixel_cluster_fibrosis_df_dict[key] = pixels_df
    

In [ ]:
metrics = {}

In [ ]:

def compute_cluster_enrichment(df, pixel_clusters, labels, column_names, label_name): 

    data = df[column_names].dropna().copy()

    label_prevalence = (
        data[column_names[1]]
        .value_counts(normalize=True)
        .reindex(labels, fill_value=0)
    )

    rows = []

    for cluster in pixel_clusters:

        cluster_data = data[data[column_names[0]] == cluster]

        proportions = (
            cluster_data[column_names[1]]
            .value_counts(normalize=True)
            .reindex(labels, fill_value=0)
        )

        row = {
            column_names[0]: cluster,
            "count": len(cluster_data),
        }

        for label in labels:

            fraction = proportions[label]
            prevalence = label_prevalence[label]

            if prevalence > 0:
                enrichment = fraction / prevalence
                log2_enrichment = np.log2(enrichment)
            else:
                enrichment = np.nan
                log2_enrichment = np.nan

            name = label_name[label]

            row[f"{name}_fraction"] = fraction
            row[f"{name}_prevalence"] = prevalence
            row[f"{name}_enrichment"] = enrichment
            row[f"{name}_log2_enrichment"] = log2_enrichment

        rows.append(row)

    return pd.DataFrame(rows)


In [ ]:
for key in keys:
    metrics[key] = {}
    metrics[key]['GS_enrichment'] = compute_cluster_enrichment(pixel_cluster_label_df_dict[key], [4, 5], [0, 2], ['pixel_som_cluster', 
                                                            'glomerulus_label'], {0: 'Background', 1: 'GS', 2: 'NGS'})
    metrics[key]['fibrosis_enrichment'] = compute_cluster_enrichment(pixel_cluster_fibrosis_df_dict[key], [2, 6], [0, 1], 
                                                                     ['pixel_som_cluster', 'fibrosis'], {0: 'Background', 1: 'Fibrosis'})

In [ ]:
for sample_key, metric_types in metrics.items():
    for metric_key, metric in metric_types.items():
        metric['Sample'] = sample_key
        metric.columns = metric.columns.str.replace('NGS', 'target', regex=False)
        metric.columns = metric.columns.str.replace('Fibrosis', 'target', regex=False)
        metrics[sample_key][metric_key] = metric

In [ ]:
results = pd.concat([metrics['HKT ANCA']['GS_enrichment'], metrics['HKT ANCA']['fibrosis_enrichment'], 
                     metrics['HKT CTRL 1']['GS_enrichment'], metrics['HKT CTRL 1']['fibrosis_enrichment']])

In [ ]:
results = results.drop(columns=['count', 'Background_fraction', 'Background_enrichment', 'target_fraction', 'target_enrichment', 'Background_prevalence', 'target_prevalence'])

In [ ]:

results["pixel_som_cluster"] = results["pixel_som_cluster"].astype(str)

samples = results["Sample"].unique()
clusters = results["pixel_som_cluster"].unique()

palette = dict(zip(
    clusters,
    sns.color_palette("tab10", n_colors=len(clusters))
))

plt.rcParams['font.size'] = 20

fig, axes = plt.subplots(
    1, len(samples),
    figsize=(5 * len(samples), 5),
    sharey=True
)

width = 0.2

for ax, sample in zip(axes, samples):
    print(sample)

    sub = results[results["Sample"] == sample].copy()

    x = np.arange(len(sub))

    # Background
    ax.bar(
        x - width / 2,
        sub["Background_log2_enrichment"],
        width,
        color=[cluster_color_map_dict[sample][int(c)] for c in sub["pixel_som_cluster"]],
        hatch="//",
        edgecolor="black",
        linewidth=0.7,
        label="Background"
    )

    # Target
    ax.bar(
        x + width / 2,
        sub["target_log2_enrichment"],
        width,
        color=[cluster_color_map_dict[sample][int(c)] for c in sub["pixel_som_cluster"]],
        edgecolor="black",
        linewidth=0.7,
        label="Target"
    )

    ax.set_xticks(x)
    ax.set_xticklabels(sub["pixel_som_cluster"])
    ax.set_title(str(sample))
    ax.set_xlabel("Cluster")
    ax.axhline(0, color="black", linewidth=0.8)

    ax.set_ylabel("log(enrichment)")

# Legend for background vs target
condition_handles = [
    plt.Rectangle((0, 0), 1, 1, facecolor="white",
                  edgecolor="black", hatch="//", label="Background"),
    plt.Rectangle((0, 0), 1, 1, facecolor="white",
                  edgecolor="black", label="Target")
]

fig.legend(
    handles=condition_handles,
    loc="upper left",
    ncol=2,
    bbox_to_anchor=(0.5, 1.2)
)

# Cluster legend
cluster_handles = [
    plt.Rectangle((0, 0), 1, 1, color=cluster_color_map_dict[list(cluster_color_map_dict.keys())[0]][int(c)], label=c)
    for c in clusters
]

fig.legend(
    handles=cluster_handles,
    title="Cluster",
    loc="upper right",
    bbox_to_anchor=(0.5, 1.2),
    ncol=min(len(clusters), 6)
)

plt.tight_layout()
plt.show()
